# BDG2 Data Preparation – Multi-Building Commercial Dataset for WattWise

**Goal:** extend the training data from a single grocery store to hundreds of real commercial buildings, converted to the same format as `adrenalin.1h.L07.B01.EM01.P.csv` so the existing model can use them without changes.

**Source:** Building Data Genome Project 2 (Miller et al., *Scientific Data*, 2020) – hourly electricity meter data, 2016–2017.

**Output:** `data/bdg2/processed/` → one CSV per building + `buildings_info.csv`

## 0. Setup
Install the required libraries (`tzdata` is needed for timezone conversion).

In [2]:
%pip install pandas tzdata


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Building selection
Keep only commercial building types (offices, healthcare, warehouses, retail, manufacturing, services, technology, food) that have an electricity meter. Universities, public and residential buildings are excluded.

In [3]:
import pandas as pd
from pathlib import Path

BDG2_DIR = Path("bdg2")
metadata = pd.read_csv(BDG2_DIR / "metadata.csv")

business_types = ["Office", "Healthcare", "Warehouse/storage", "Retail",
                  "Manufacturing/industrial", "Services", "Technology/science",
                  "Food sales and service"]

selected = metadata[
    metadata.primaryspaceusage.isin(business_types) & (metadata.electricity == "Yes")
]
selected.primaryspaceusage.value_counts()

primaryspaceusage
Office                      296
Healthcare                   27
Warehouse/storage            14
Retail                       11
Manufacturing/industrial     11
Services                      9
Technology/science            7
Food sales and service        6
Name: count, dtype: int64

## 2. Load electricity data
Load only the columns of the selected buildings to keep memory usage low.

In [4]:
header = pd.read_csv(BDG2_DIR / "electricity_cleaned.csv", nrows=0).columns
building_ids = [b for b in selected.building_id if b in header]

electricity = pd.read_csv(
    BDG2_DIR / "electricity_cleaned.csv",
    usecols=["timestamp"] + building_ids,
    parse_dates=["timestamp"],
)
print(electricity.shape)

(17544, 382)


In [5]:
print(electricity.head())

            timestamp  Panther_office_Hannah  Panther_retail_Felix  \
0 2016-01-01 00:00:00                    NaN                   NaN   
1 2016-01-01 01:00:00                    NaN                   NaN   
2 2016-01-01 02:00:00                    NaN                   NaN   
3 2016-01-01 03:00:00                    NaN                   NaN   
4 2016-01-01 04:00:00                    NaN                   NaN   

   Panther_retail_Gilbert  Panther_office_Catherine  Panther_office_Patti  \
0                     NaN                       NaN                   NaN   
1                     NaN                       NaN                   NaN   
2                     NaN                       NaN                   NaN   
3                     NaN                       NaN                   NaN   
4                     NaN                       NaN                   NaN   

   Panther_retail_Romeo  Panther_office_Lauretta  Panther_office_Valarie  \
0                   NaN                 

## 3. Cleaning settings
- Gaps of **≤ 3 hours** are filled with linear interpolation.
- Buildings with **> 20% missing values** after filling are discarded.

In [6]:
OUTPUT_DIR = BDG2_DIR / "processed"
OUTPUT_DIR.mkdir(exist_ok=True)

timezones = selected.set_index("building_id").timezone
elec = electricity.set_index("timestamp")

MAX_GAP_HOURS = 3         # fill gaps up to 3 hours
MAX_MISSING_SHARE = 0.20  # skip buildings with more than 20% missing

## 4. Cleaning walkthrough on one building
Step-by-step example on `Panther_office_Hannah`.

### 4.1 Trim empty periods
Remove hours before the meter was installed and after it stopped recording.

In [7]:
# Pick one building to inspect the cleaning step by step
example = "Panther_office_Hannah"

# Select that building's column: a Series of hourly kWh values,
# indexed by timestamp (thanks to set_index in Block 1)
series = elec[example]

# How many hours in total, and how many are empty (NaN)?
print("Total hours:", len(series), "| empty:", series.isna().sum())

# first_valid_index() -> timestamp of the first real (non-NaN) value
# last_valid_index()  -> timestamp of the last real value
# .loc[start:end] keeps only the rows between those two timestamps
series = series.loc[series.first_valid_index():series.last_valid_index()]

print("After trimming:", len(series), "| empty:", series.isna().sum())
print("Data starts:", series.index.min(), "| ends:", series.index.max())

Total hours: 17544 | empty: 3390
After trimming: 14621 | empty: 467
Data starts: 2016-05-01 19:00:00 | ends: 2017-12-31 23:00:00


### 4.2 Fill short gaps
Interpolate gaps of up to 3 hours. Longer gaps are left empty: filling them would mean inventing data.

In [8]:
# True where the value is missing, False where it exists
is_missing = series.isna()

# Give each consecutive run of missing/present hours its own ID.
# The ID increases every time we switch from "present" to "missing" (or back),
# so all hours belonging to the same gap share one ID.
gap_id = (is_missing != is_missing.shift()).cumsum()

# For every hour, count how many missing hours are in its run
# -> each missing hour now knows the length of the gap it belongs to
gap_length = is_missing.groupby(gap_id).transform("sum")

# Only hours that are missing AND belong to a gap of <= 3 hours
short_gap = is_missing & (gap_length <= MAX_GAP_HOURS)

# interpolate() fills NaNs with a straight line between neighbours.
# where(~short_gap, ...) keeps original values everywhere EXCEPT short-gap hours,
# which take the interpolated value. Long gaps stay NaN.
print("Short-gap hours to fill:", short_gap.sum())
series = series.where(~short_gap, series.interpolate())

print("Empty hours left (long gaps):", series.isna().sum())

Short-gap hours to fill: 2
Empty hours left (long gaps): 465


### 4.3 Quality check
Discard the building if more than 20% of its hours are still missing.

In [9]:
# .mean() on True/False gives the share of True values
# -> fraction of hours still missing (0.05 = 5%)
missing_share = series.isna().mean()
print(f"Missing after filling: {missing_share:.1%}")

if missing_share > MAX_MISSING_SHARE:
    print("Building DISCARDED: too many missing values")
else:
    # Remove hours that are still empty (long gaps).
    # The series will have time "jumps" where those gaps were.
    series = series.dropna()
    print("Building KEPT, valid hours:", len(series))

Missing after filling: 3.2%
Building KEPT, valid hours: 14156


### 4.4 Convert to the model format
Convert local time to UTC and rename the columns to match Adrian's dataset.

In [10]:
# BDG2 timestamps are LOCAL time with no timezone attached.
# tz_localize() tells pandas which timezone they are in (e.g. US/Eastern).
#   ambiguous="NaT"   -> repeated hour when clocks go back (DST end) becomes empty
#   nonexistent="NaT" -> skipped hour when clocks go forward (DST start) becomes empty
# tz_convert("UTC") shifts every timestamp to UTC, like Adrian's data.
utc_time = (
    series.index
    .tz_localize(timezones[example], ambiguous="NaT", nonexistent="NaT")
    .tz_convert("UTC")
)

output = pd.DataFrame({
    # strftime formats the date as text in the same style as Adrian's file
    "timestamp [UTC]": utc_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "value P [Total active power (kW)]": series.values,
}).dropna()  # removes the few DST hours that became empty

# Compare with Adrian's file: column names must match
print(output.head(3))
print(pd.read_csv("adrenalin.1h.L07.B01.EM01.P.csv").head(3))

        timestamp [UTC]  value P [Total active power (kW)]
0  2016-05-01T23:00:00Z                            15.2397
1  2016-05-20T22:00:00Z                             5.0440
2  2016-05-20T23:00:00Z                             6.6763
        timestamp [UTC]  value P [Total active power (kW)]
0  2021-11-29T06:00:00Z                          52.400002
1  2021-11-29T07:00:00Z                          69.812500
2  2021-11-29T08:00:00Z                          55.779167


## 5. Process all buildings
Apply steps 4.1–4.4 to every selected building and save the results.

**Result:** 368 buildings saved, 13 discarded.

In [11]:
def clean_building(building):
    """Clean one building and return it in Adrian's format,
    or None if it has too many missing values."""
    s = elec[building]

    # Block 2: trim to first/last real value
    s = s.loc[s.first_valid_index():s.last_valid_index()]

    # Block 3: fill only gaps of <= 3 hours
    miss = s.isna()
    length = miss.groupby((miss != miss.shift()).cumsum()).transform("sum")
    s = s.where(~(miss & (length <= MAX_GAP_HOURS)), s.interpolate())

    # Block 4: discard unreliable buildings, drop remaining gaps
    if s.isna().mean() > MAX_MISSING_SHARE:
        return None
    s = s.dropna()

    # Block 5: local time -> UTC, same columns as Adrian's file
    utc = (s.index
           .tz_localize(timezones[building], ambiguous="NaT", nonexistent="NaT")
           .tz_convert("UTC"))
    return pd.DataFrame({
        "timestamp [UTC]": utc.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "value P [Total active power (kW)]": s.values,
    }).dropna()


kept, skipped = [], []
for building in building_ids:
    result = clean_building(building)
    if result is None:
        skipped.append(building)  # too many missing values
    else:
        # One file per building, e.g. bdg2/processed/Panther_office_Hannah.csv
        result.to_csv(OUTPUT_DIR / f"{building}.csv", index=False)
        kept.append(building)

# Info table: one row per kept building (type, size, location, timezone).
# Useful for the model (building type as a feature) and for the paper.
info = selected[selected.building_id.isin(kept)][
    ["building_id", "site_id", "primaryspaceusage", "sub_primaryspaceusage", "sqm", "timezone"]
]
info.to_csv(OUTPUT_DIR / "buildings_info.csv", index=False)

print(f"Saved {len(kept)} buildings, skipped {len(skipped)}")
print(info.primaryspaceusage.value_counts())

Saved 368 buildings, skipped 13
primaryspaceusage
Office                      287
Healthcare                   25
Warehouse/storage            13
Retail                       11
Manufacturing/industrial     11
Services                      9
Food sales and service        6
Technology/science            6
Name: count, dtype: int64


## 6. Notes for the team
- Download BDG2 and place the files in `data/bdg2/` (excluded from Git).
- Long gaps create time "jumps" in the series: lag features must handle them.
- `buildings_info.csv` contains the building type, which can be used as a model feature.